# 02 — Mocking avec `unittest.mock`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre **quand** et **pourquoi** mocker
- utiliser `MagicMock` pour créer des objets factices
- utiliser `patch` pour remplacer une dépendance pendant un test
- maîtriser `spec`, `side_effect`, `return_value`

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- pytest, fixtures, parametrize
- classes, Protocol, injection de dépendance

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- factory_boy (notebook 03)

## Plan

1. Pourquoi mocker
2. `MagicMock`
3. `patch` — remplacer un objet
4. `spec` — mock fidèle à l'interface
5. `side_effect` et `return_value`
6. `patch` comme décorateur, CM, fixture
7. Quand ne pas mocker
8. Synthèse
9. Exercices

---

## 1. Pourquoi mocker

Vous voulez tester une fonction qui envoie un email, appelle une API, ou interroge une base de données. En test, vous ne voulez pas **vraiment** envoyer l'email. Un **mock** remplace la dépendance réelle par un faux, contrôlé par le test.

---

## 2. `MagicMock`

`MagicMock` crée un objet qui accepte **tout appel** et retourne un autre `MagicMock`.

In [ ]:
from unittest.mock import MagicMock

m = MagicMock()
m.methode(1, 2, 3)
m.attribut.sous_attribut


In [ ]:
m.methode.assert_called_once_with(1, 2, 3)


In [ ]:
m.methode.call_count


---

## 3. `patch` — remplacer un objet

`patch` remplace temporairement un objet (fonction, classe, constante) par un mock.

In [ ]:
from unittest.mock import patch


def obtenir_heure() -> str:
    import datetime
    return datetime.datetime.now().strftime('%H:%M')


def test_heure():
    with patch('datetime.datetime') as mock_dt:
        mock_dt.now.return_value.strftime.return_value = '14:30'
        assert obtenir_heure() == '14:30'


---

## 4. `spec` — mock fidèle à l'interface

Avec `spec`, le mock **lève une erreur** si on appelle un attribut qui n'existe pas sur l'original. Ça empêche les tests de passer par accident.

In [ ]:
class Notifieur:
    def envoyer(self, msg: str) -> None: ...

m = MagicMock(spec=Notifieur)
m.envoyer('test')  # OK

try:
    m.methode_inexistante()
except AttributeError as exc:
    print(exc)


---

## 5. `side_effect` et `return_value`

In [ ]:
m = MagicMock()
m.return_value = 42
m()  # → 42


In [ ]:
m.side_effect = ValueError('boom')
try:
    m()
except ValueError as exc:
    print(exc)


In [ ]:
m.side_effect = [1, 2, 3]  # retours successifs
m(), m(), m()


---

## 6. `patch` comme décorateur, CM, fixture

In [ ]:
# Comme décorateur
@patch('builtins.print')
def test_silencieux(mock_print):
    print('invisible')
    mock_print.assert_called_once_with('invisible')


In [ ]:
# Comme fixture pytest (via monkeypatch ou conftest)
import pytest

@pytest.fixture
def mock_notifieur():
    with patch('__main__.Notifieur', spec=Notifieur) as m:
        yield m


---

## 7. Quand ne pas mocker

- **Ne mockez pas les structures de données** : créez-les pour de vrai.
- **Ne mockez pas les algorithmes à tester** : si vous mockez tout, vous testez le mock.
- **Préférez l'injection de dépendance** (Protocol + fake) au mocking lourd.
- **Trop de mocking = signe d'un couplage trop fort.**

---

## Synthèse

| Outil | Rôle |
|---|---|
| `MagicMock()` | Faux objet accepte tout |
| `MagicMock(spec=X)` | Faux fidèle à l'interface de X |
| `patch('module.nom')` | Remplace temporairement |
| `return_value` | Valeur retournée par le mock |
| `side_effect` | Exception ou séquence de retours |
| `assert_called_once_with(...)` | Vérification d'appel |


### Règles à retenir

1. **Mocker au bord** : les appels réseau, filesystem, horloge.
2. **`spec=True` toujours** pour éviter les faux positifs.
3. **Injection de dépendance > mocking** quand c'est possible.
4. **Un test qui mocke 5 choses est un signal d'alarme.**

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — MagicMock de base *(facile)*

Créer un `MagicMock(spec=list)`. Appeler `.append(42)`. Vérifier avec `assert_called_once_with(42)`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Mocking", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from unittest.mock import MagicMock

m = MagicMock(spec=list)
m.append(42)
m.append.assert_called_once_with(42)
print('ok')
```

</details>

### Exercice 2 — Patcher `time.time` *(moyen)*

Écrire une fonction `mesurer(func)` qui renvoie le temps d'exécution en secondes. Tester en patchant `time.perf_counter` pour qu'il retourne 10.0 puis 10.5, et vérifier que `mesurer` renvoie 0.5.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Mocking", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
from unittest.mock import patch

def mesurer(func) -> float:
    t0 = time.perf_counter()
    func()
    return time.perf_counter() - t0

def test_mesurer():
    with patch('time.perf_counter', side_effect=[10.0, 10.5]):
        assert mesurer(lambda: None) == 0.5

test_mesurer()
print('ok')
```

</details>

### Exercice 3 — Service avec mock de DB *(difficile)*

Écrire un `Service` qui reçoit un `Protocol BaseDonnees` avec `charger(id) -> dict | None`. Écrire un test qui mocke `BaseDonnees` et vérifie que `Service.obtenir(id)` appelle bien `charger` et gère le cas `None`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Mocking", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol
from unittest.mock import MagicMock

class BaseDonnees(Protocol):
    def charger(self, id: int) -> dict | None: ...

class Service:
    def __init__(self, db: BaseDonnees) -> None:
        self.db = db
    def obtenir(self, id: int) -> dict:
        result = self.db.charger(id)
        if result is None:
            raise ValueError(f'id {id} introuvable')
        return result

def test_obtenir_ok():
    db = MagicMock(spec=BaseDonnees)
    db.charger.return_value = {'nom': 'Alice'}
    assert Service(db).obtenir(1) == {'nom': 'Alice'}
    db.charger.assert_called_once_with(1)

def test_obtenir_not_found():
    db = MagicMock(spec=BaseDonnees)
    db.charger.return_value = None
    import pytest
    with pytest.raises(ValueError, match='introuvable'):
        Service(db).obtenir(99)

test_obtenir_ok()
test_obtenir_not_found()
print('ok')
```

</details>

---

## Ressources externes

### Documentation officielle
- [`unittest.mock`](https://docs.python.org/3/library/unittest.mock.html)

### Lectures complémentaires
- *Architecture Patterns with Python* — chapitre sur les fakes et les mocks.